# Week 3 • Day 3: Tokenizers & Chat Templates (Lec 72–75)

Welcome to Day 3 of Week 3! In this notebook, we explore how Large Language Models convert raw human text into numbers (tokens), how chat templates format conversations, and how different model families (Llama 3.1, Phi-4, DeepSeek, and QwenCoder) compare in tokenization efficiency.

## 📦 Step 1: Environment Setup & Dependencies

Run the cell below to install the required libraries. This notebook can run entirely on a **free CPU** or on a T4 GPU.

In [ ]:
# Install Hugging Face transformers, datasets, and hub
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6 huggingface_hub


In [ ]:
import os
from huggingface_hub import login
from transformers import AutoTokenizer

# Attempt login via Colab Secrets (HF_TOKEN) or prompt
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.getenv('HF_TOKEN')

if hf_token and hf_token.startswith('hf_'):
    print('✅ Hugging Face token detected!')
    login(hf_token, add_to_git_credential=True)
else:
    print('ℹ️ HF_TOKEN not found in secrets. You can still run ungated models, or call login() manually.')


---
# 🧠 Lecture 72: Tokenizers — How LLMs Convert Text to Numbers

LLMs do not understand English or characters directly. They perform math on **Token IDs** (integers).
Here we load the tokenizer for Meta Llama 3.1 8B (or an open fallback if you have not accepted Meta's terms yet).

In [ ]:
# Load Llama 3.1 Tokenizer (requires agreed terms on HF: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B)
# Fallback model if access pending: 'Qwen/Qwen2.5-7B' or 'gpt2'
MODEL_ID = 'meta-llama/Meta-Llama-3.1-8B'

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    print(f'✅ Successfully loaded tokenizer: {MODEL_ID}')
except Exception as e:
    print(f'⚠️ Could not access {MODEL_ID} (likely gated). Falling back to Qwen2.5 tokenizer for demo:', e)
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B', trust_remote_code=True)


In [ ]:
# 1. Encode: Convert raw text to token IDs
text = 'I am excited to show Tokenizers in action to my LLM engineers'
tokens = tokenizer.encode(text)
print('Token IDs:', tokens)


In [ ]:
# 2. Compare character, word, and token counts
character_count = len(text)
word_count = len(text.split(' '))
token_count = len(tokens)
print(f'There are {character_count} characters, {word_count} words and {token_count} tokens.')
print(f'Compression: {character_count / token_count:.2f} chars/token | {token_count / word_count:.2f} tokens/word')


In [ ]:
# 3. Decode: Convert token IDs back to a full string
print('Full Decoded:', tokenizer.decode(tokens))

# 4. Batch Decode: See each individual token piece as a string!
print('\nIndividual Tokens:')
for tok_id, tok_str in zip(tokens, tokenizer.batch_decode(tokens)):
    print(f'{tok_id:>7} -> {repr(tok_str)}')


In [ ]:
# 5. Vocabulary Size & Special Added Tokens
print('Total Vocabulary Size:', len(tokenizer.vocab))
print('\nSample Added/Special Vocab:')
added = tokenizer.get_added_vocab()
for k, v in list(added.items())[:10]:
    print(f'{k:>30} -> ID: {v}')


---
# 💬 Lecture 73 & 74: Instruct Models & The Chat Template "Aha!" Moment

### The Crucial "Aha!" Moment
For weeks, we have written messages like:
```python
messages = [
    {'role': 'system', 'content': 'You are a helpful assistant'},
    {'role': 'user', 'content': 'Tell a light-hearted joke for Data Scientists'}
]
```
**An LLM is a blind statistical model that predicts the next token ID from an input sequence of numbers.** It cannot accept Python dictionaries!

`tokenizer.apply_chat_template()` converts this list of dicts into the exact string representation the model was trained on, inserting special delimiter tags like `<|start_header_id|>`, `<|eot_id|>`, etc.

In [ ]:
# Load Instruct Tokenizer
try:
    instruct_tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)
except Exception:
    instruct_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B-Instruct', trust_remote_code=True)

messages = [
    {'role': 'system', 'content': 'You are a helpful assistant'},
    {'role': 'user', 'content': 'Tell a light-hearted joke for a room of Data Scientists'}
]

# Convert to raw string using the model's trained Chat Template
prompt = instruct_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print('=== EXACT STRING SENT TO THE NEURAL NETWORK ===')
print(prompt)


> **Notice the end of the prompt above:**  
> It ends with `<|start_header_id|>assistant<|end_header_id|>\n\n`!  
> Because `add_generation_prompt=True`, the tokenizer leaves the assistant prompt open so the model's next token prediction will naturally complete the assistant's reply!

---
# ⚔️ Lecture 75: Comparing Tokenizers — Phi-4, DeepSeek & QwenCoder

Now we compare 4 major model tokenizers:
1. **Meta Llama 3.1 8B** (`meta-llama/Meta-Llama-3.1-8B-Instruct`)
2. **Microsoft Phi-4** (`microsoft/Phi-4-mini-instruct`)
3. **DeepSeek** (`deepseek-ai/DeepSeek-V3.1` or `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`)
4. **Qwen 2.5 Coder** (`Qwen/Qwen2.5-Coder-7B-Instruct`)


In [ ]:
PHI4 = 'microsoft/Phi-4-mini-instruct'
DEEPSEEK = 'deepseek-ai/DeepSeek-V3.1'
QWEN_CODER = 'Qwen/Qwen2.5-Coder-7B-Instruct'

print('Downloading and loading tokenizers...')
phi4_tok = AutoTokenizer.from_pretrained(PHI4, trust_remote_code=True)
deepseek_tok = AutoTokenizer.from_pretrained(DEEPSEEK, trust_remote_code=True)
qwen_tok = AutoTokenizer.from_pretrained(QWEN_CODER, trust_remote_code=True)
print('✅ All tokenizers loaded successfully!')


In [ ]:
# Compare tokenization on English text
text = 'I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers'

print(f'Input Text: "{text}"\n')
print('Llama Tokens:', tokenizer.encode(text))
print('Phi-4 Tokens:', phi4_tok.encode(text))
print('DeepSeek Tokens:', deepseek_tok.encode(text))
print('QwenCoder Tokens:', qwen_tok.encode(text))


In [ ]:
# Compare Chat Templates
print('=== Llama 3.1 Chat Template ===')
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

print('\n=== Microsoft Phi-4 Chat Template ===')
print(phi4_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

print('\n=== DeepSeek Chat Template ===')
print(deepseek_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))


In [ ]:
# 🧑‍💻 Code Tokenization Benchmark: Why QwenCoder is a Superpower for Code!
code_snippet = '''
def hello_world(person):
    print("Hello", person)
'''

qwen_code_tokens = qwen_tok.encode(code_snippet)
print(f'QwenCoder Code Tokens (Total = {len(qwen_code_tokens)}):\n')
for tok_id in qwen_code_tokens:
    print(f'{tok_id:>7} = {repr(qwen_tok.decode(tok_id))}')


### 🎉 Congratulations on Completing Week 3 Day 3!
You now understand:
1. How text is converted into integers via Byte-Pair Encoding.
2. Why chat templates are the real bridge between Python objects and neural networks.
3. How specialized tokenizers like QwenCoder optimize code syntax and whitespace for maximum efficiency!